# Robustness and interpretability analysis for AstroLens GCNN

Adapts two analyses from the GCNN paper's reference implementation
([`GCNNMorphology`](https://github.com/snehjp2/GCNNMorphology),
`src/scripts/onepixelattack.py` and `src/scripts/latent_space_analysis.py`)
to `astrolens.models.gcnn.GCNN` and its registry:

- **One-pixel attack**: a black-box adversarial attack (Su et al., 2019) that
  searches for a single pixel whose color change flips a model's prediction,
  using differential evolution
  ([`scipy.optimize.differential_evolution`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.differential_evolution.html))
  instead of the reference implementation's `escnn`/CMA-ES-based search. Run
  against **both** `gcnn_d4` (dihedral group `D4`, rotation/reflection
  equivariant) and `gcnn_c1` (trivial group `C1`, order 1 — architecturally
  an ordinary CNN with the same block structure but no equivariance
  constraint) — a GCNN-only run can't show whether a low attack success rate
  reflects genuine equivariant robustness or just that one-pixel attacks are
  weak against this dataset/architecture family in general.
- **Latent-space analysis**: a t-SNE projection ([`sklearn.manifold.TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html),
  in place of the reference implementation's UMAP, to reuse the
  `scikit-learn` dependency already used for the train/val/test split) of
  `gcnn_d4`'s invariant embedding (`forward_features` + group pooling,
  before the classification head), colored by galaxy class.

Requires a checkpoint from `examples/gz10_gcnn_training.ipynb`
(`gcnn_d4.pt`) — run that notebook first. `gcnn_c1` is trained from scratch
in this notebook (same recipe, ~15x fewer FLOPs/params than `gcnn_d4`).

## Install example-only dependencies

Not part of AstroLens' core install (`requirements.txt`) — only needed for this example.

In [ ]:
!pip install -q datasets torchvision scikit-learn scipy

## Imports

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from PIL import Image
from scipy.optimize import differential_evolution
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

import astrolens

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load the dataset and the trained GCNN checkpoint

Same 70/10/20 split (`random_state=0`) and augmentation as `gz10_gcnn_training.ipynb`, so `test_idx` reproduces the same held-out images the checkpoint was evaluated on, and `train_idx`/`val_idx` let this notebook train the `gcnn_c1` baseline under the identical recipe.

In [ ]:
IMG_SIZE = 255
CHECKPOINT_PATH = "gcnn_d4.pt"
CLASS_NAMES = [
    "disturbed",
    "merging",
    "round_smooth",
    "in_between_round_smooth",
    "cigar_shaped_smooth",
    "barred_spiral",
    "unbarred_tight_spiral",
    "unbarred_loose_spiral",
    "edge_on_no_bulge",
    "edge_on_with_bulge",
]
NUM_CLASSES = len(CLASS_NAMES)

gz10 = load_dataset("UniverseTBD/mmu_gz10", split="train")
labels = gz10["gz10_label"]

train_idx, rest_idx = train_test_split(
    range(len(gz10)), train_size=0.7, stratify=labels, random_state=0
)
val_idx, test_idx = train_test_split(
    rest_idx,
    train_size=1 / 3,
    stratify=[labels[i] for i in rest_idx],
    random_state=0,
)

# reference implementation's augmentation and normalization (src/scripts/train.py)
train_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.RandomRotation(180),
        transforms.Resize(IMG_SIZE),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(IMG_SIZE),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
)


class GZ10Dataset(Dataset):
    """Map-style wrapper around an index subset of the HF split, applying transform lazily."""

    def __init__(self, hf_split, indices, transform):
        self.hf_split = hf_split
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        example = self.hf_split[self.indices[i]]
        image = Image.open(io.BytesIO(example["rgb_image"]["bytes"])).convert("RGB")
        return self.transform(image), example["gz10_label"]


train_dataset = GZ10Dataset(gz10, train_idx, train_transform)
val_dataset = GZ10Dataset(gz10, val_idx, eval_transform)
test_dataset = GZ10Dataset(gz10, test_idx, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=4)

gcnn_d4 = astrolens.create_model(
    "gcnn_d4", img_size=IMG_SIZE, in_chans=3, num_classes=NUM_CLASSES
).to(device)
gcnn_d4.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
gcnn_d4.eval()

len(train_dataset), len(val_dataset), len(test_dataset)

## Train the `gcnn_c1` baseline

Same optimizer/schedule/class-weighting recipe as `gz10_gcnn_training.ipynb`. `gcnn_c1` uses `GCNN`'s trivial group `C1` (order 1): with no rotation or reflection elements, `GroupConv2d` reduces to an ordinary convolution — the same masked, anti-aliased conv-block stack as `gcnn_d4`, but without the equivariance constraint, so it isolates that constraint's effect on robustness rather than comparing across unrelated architectures.

In [ ]:
MAX_EPOCHS = 10
LR = 1e-2
WEIGHT_DECAY = 1e-4
MILESTONES = [round(MAX_EPOCHS * f) for f in (0.25, 0.5, 0.75)]
GAMMA = 0.1

train_counts = torch.bincount(
    torch.tensor([labels[i] for i in train_idx]), minlength=NUM_CLASSES
).float()
class_weights = (train_counts.sum() / (NUM_CLASSES * train_counts)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)


def run_epoch(model, loader, train: bool, optimizer=None):
    model.train(train)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, batch_labels in loader:
            images, batch_labels = images.to(device), batch_labels.to(device)
            logits = model(images)
            loss = criterion(logits, batch_labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            count += images.size(0)

    return total_loss / count, correct / count


gcnn_c1 = astrolens.create_model(
    "gcnn_c1", img_size=IMG_SIZE, in_chans=3, num_classes=NUM_CLASSES
).to(device)
optimizer = torch.optim.AdamW(gcnn_c1.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=MILESTONES, gamma=GAMMA)

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc = run_epoch(gcnn_c1, train_loader, train=True, optimizer=optimizer)
    val_loss, val_acc = run_epoch(gcnn_c1, val_loader, train=False)
    scheduler.step()
    print(
        f"epoch {epoch}/{MAX_EPOCHS} "
        f"train_loss={train_loss:.3f} train_acc={train_acc:.3f} "
        f"val_loss={val_loss:.3f} val_acc={val_acc:.3f}"
    )

gcnn_c1.eval()
test_loss, test_acc = run_epoch(gcnn_c1, test_loader, train=False)
print(f"gcnn_c1 test_acc={test_acc:.3f}")

## One-pixel attack: `gcnn_d4` vs. `gcnn_c1`

For each candidate image, `differential_evolution` searches over one pixel's `(x, y, r, g, b)` for the setting that minimizes a model's softmax probability on the image's true class — an attack succeeds if that probability drops below every other class's. Both models are attacked on the same images, restricted to ones both classify correctly, so the comparison isolates the effect of equivariance rather than differences in which images each model already gets right.

In [ ]:
def perturb(image: torch.Tensor, x) -> torch.Tensor:
    """Apply one pixel edit `x = (px, py, r, g, b)` (all in [0, 1]) to a copy of `image`."""
    _, h, w = image.shape
    px, py = round(x[0] * (w - 1)), round(x[1] * (h - 1))
    out = image.clone()
    out[:, py, px] = image.new_tensor(x[2:]) * 2 - 1  # [0, 1] -> normalized [-1, 1]
    return out


@torch.no_grad()
def true_class_prob(x, model, image: torch.Tensor, true_label: int) -> float:
    logits = model(perturb(image, x).unsqueeze(0).to(device))
    return torch.softmax(logits, dim=1)[0, true_label].item()


def one_pixel_attack(model, image: torch.Tensor, true_label: int, maxiter=30, popsize=15):
    bounds = [(0, 1)] * 5
    result = differential_evolution(
        true_class_prob,
        bounds,
        args=(model, image, true_label),
        maxiter=maxiter,
        popsize=popsize,
        tol=1e-4,
        seed=0,
        polish=False,
    )
    return perturb(image, result.x), result.x


N_ATTACK_IMAGES = 5

correct_images = []
with torch.no_grad():
    for images, batch_labels in test_loader:
        images = images.to(device)
        pred_d4 = gcnn_d4(images).argmax(dim=1).cpu()
        pred_c1 = gcnn_c1(images).argmax(dim=1).cpu()
        for image, label, p_d4, p_c1 in zip(images.cpu(), batch_labels, pred_d4, pred_c1):
            if label == p_d4 == p_c1:
                correct_images.append((image, label.item()))
        if len(correct_images) >= N_ATTACK_IMAGES:
            break
correct_images = correct_images[:N_ATTACK_IMAGES]

attack_results = {"gcnn_d4": [], "gcnn_c1": []}
for name, model in [("gcnn_d4", gcnn_d4), ("gcnn_c1", gcnn_c1)]:
    for image, true_label in correct_images:
        adversarial, _ = one_pixel_attack(model, image, true_label)
        with torch.no_grad():
            adv_pred = model(adversarial.unsqueeze(0).to(device)).argmax(dim=1).item()
        attack_results[name].append((image, adversarial, true_label, adv_pred))

for name, results in attack_results.items():
    success_rate = sum(t != a for _, _, t, a in results) / len(results)
    print(f"{name} one-pixel attack success rate: {success_rate:.0%} ({len(results)} images)")

In [ ]:
def show_image(ax, image: torch.Tensor, title: str):
    npimg = (image.cpu().numpy() * 0.5 + 0.5).clip(0, 1)  # undo Normalize(0.5, 0.5, 0.5)
    ax.imshow(np.transpose(npimg, (1, 2, 0)))
    ax.set_title(title, fontsize=9)
    ax.axis("off")


fig, axes = plt.subplots(4, N_ATTACK_IMAGES, figsize=(3 * N_ATTACK_IMAGES, 12))
for row, name in enumerate(["gcnn_d4", "gcnn_c1"]):
    for i, (image, adversarial, true_label, adv_pred) in enumerate(attack_results[name]):
        show_image(axes[2 * row, i], image, f"{name} orig: {CLASS_NAMES[true_label]}")
        show_image(axes[2 * row + 1, i], adversarial, f"{name} pert: {CLASS_NAMES[adv_pred]}")
fig.tight_layout()

## Latent-space analysis

`GCNN.forward_features` returns the regular-representation features before invariant group pooling; applying `model.gpool` and flattening gives the same rotation/reflection-invariant embedding the classification head consumes. Shown for `gcnn_d4`, the model this invariance is meaningful for.

In [ ]:
@torch.no_grad()
def embed(model, images: torch.Tensor) -> torch.Tensor:
    features = model.forward_features(images.to(device))
    return model.gpool(features).flatten(1)


embeddings, embedding_labels = [], []
for images, batch_labels in test_loader:
    embeddings.append(embed(gcnn_d4, images).cpu())
    embedding_labels.append(batch_labels)
embeddings = torch.cat(embeddings).numpy()
embedding_labels = torch.cat(embedding_labels).numpy()

projection = TSNE(n_components=2, random_state=0, init="pca").fit_transform(embeddings)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(
    projection[:, 0], projection[:, 1], c=embedding_labels, cmap="tab10", s=6, alpha=0.7
)
handles, _ = scatter.legend_elements(num=NUM_CLASSES)
ax.legend(handles, CLASS_NAMES, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.set_title("t-SNE of gcnn_d4 invariant embeddings (test split)")
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()